# 07 — AI-Era Analysis: Interrupted Time Series + Difference-in-Differences  
### *Coding Alone* · ICSE 2027 SEIS pipeline

**Self-contained.** Set PROJECT_ID → **Runtime → Run all**. Reads your metric tables (cheap).

Tests the paper's central question: *beyond the secular trends and the 2020 pandemic shift, is there an AI-era change in the isolation signals?* Three complementary analyses:

1. **Interrupted Time Series (ITS)** — segmented regression on each ecosystem series with **both** a 2020 COVID break and a 2022-11 AI break, HAC (Newey–West) standard errors. Isolates the AI-era level/slope change *after* accounting for the pandemic shift and the secular slope.
2. **Difference-in-Differences (DiD)** — high-AI-exposure languages (Python/JS/TS — where LLM assistance is strongest) vs lower-exposure languages, two-way fixed effects, treatment = high×post-2022. The DiD coefficient is the incremental AI-era effect net of common trends.
3. **Placebo + parallel-trends** checks — a fake 2021 break (should be null) and a pre-2022 trend test (should be flat) to keep the DiD honest.

> This is a proxy identification (AI tools aren't strictly language-bound; LLMs are simply *more capable* on high-resource languages). Limitations noted in the output and to be stated in the paper.


## Config

In [ ]:
# ==== EDIT ME ==============================================================
PROJECT_ID = "YOUR-GCP-PROJECT-ID"
# ==========================================================================
BQ_DATASET="seis"; BQ_LOCATION="US"; DRIVE_BASE="/content/drive/MyDrive/ICSE_SEIS"
WIN_START="201801"; WIN_END="202506"; MDI_END="202412"   # MDI right-censors after 2024
GAP=["202110"]
HIGH_AI = ["Python","JavaScript","TypeScript"]            # high LLM-assistance languages
COVID_YM="202003"; AI_YM="202211"; PLACEBO_YM="202106"
assert PROJECT_ID!="PASTE-YOUR-PROJECT-ID-HERE","Set PROJECT_ID first."
print("Config loaded.")


## Setup

In [ ]:
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from google.colab import auth, drive
from google.cloud import bigquery
auth.authenticate_user()
client=bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
drive.mount('/content/drive')
import os; os.makedirs(DRIVE_BASE+"/tables", exist_ok=True)
P=f"{PROJECT_ID}.{BQ_DATASET}"
def q(sql): return client.query(sql).result().to_dataframe()
def ymidx(s): return (s.str[:4].astype(int)-2018)*12 + (s.str[4:].astype(int)-1)  # months since 2018-01
print("Ready.", P)


## 1. Pull the ecosystem monthly series (for ITS)

In [ ]:
eco = q(f"""
SELECT ym,
  AVG(IF(active,cd,NULL)) AS cd, AVG(swr) AS swr,
  AVG(reciprocity) AS cr, AVG(hie_norm) AS hie
FROM `{P}.metrics_panel`
WHERE ym BETWEEN '{WIN_START}' AND '{WIN_END}' AND ym NOT IN UNNEST({GAP})
GROUP BY ym ORDER BY ym""")
nrr = q(f"""
SELECT ym, AVG(IF(responses>0,1,0)) AS nrr
FROM `{P}.mdi_newcomers`
WHERE ym BETWEEN '{WIN_START}' AND '{MDI_END}' AND ym NOT IN UNNEST({GAP})
GROUP BY ym ORDER BY ym""")
eco = eco.merge(nrr, on="ym", how="left")
eco["t"]=ymidx(eco.ym); eco["mo"]=eco.ym.str[4:]
for br,name in [(COVID_YM,"covid"),(AI_YM,"ai")]:
    b=(int(br[:4])-2018)*12+(int(br[4:])-1)
    eco[f"{name}_lvl"]=(eco.t>=b).astype(int)
    eco[f"{name}_slope"]=np.maximum(0, eco.t-b)
print(eco.shape); eco.head(3)


## 2. Interrupted Time Series — AI-era level & slope, net of COVID + secular trend
For each metric: `y ~ t + covid_lvl + covid_slope + ai_lvl + ai_slope + month-dummies`, HAC SEs (12 lags). **`ai_lvl`** = immediate level shift at ChatGPT; **`ai_slope`** = change in trend after. A real AI-era effect shows up here *beyond* the secular `t` and the COVID terms.

In [ ]:
def its(metric):
    d=eco.dropna(subset=[metric]).copy()
    m=smf.ols(f"{metric} ~ t + covid_lvl + covid_slope + ai_lvl + ai_slope + C(mo)", data=d
             ).fit(cov_type="HAC", cov_kwds={"maxlags":12})
    return {"metric":metric,
            "secular_slope":m.params["t"], "p_secular":m.pvalues["t"],
            "ai_level":m.params["ai_lvl"], "p_ai_level":m.pvalues["ai_lvl"],
            "ai_slope":m.params["ai_slope"], "p_ai_slope":m.pvalues["ai_slope"]}
its_res=pd.DataFrame([its(x) for x in ["cd","swr","cr","hie","nrr"]]).round(5)
its_res.to_csv(f"{DRIVE_BASE}/tables/its_results.csv", index=False)
its_res


## 3. Language × month panel (for DiD)

In [ ]:
lm = q(f"""
SELECT l.language, p.ym,
  AVG(IF(p.active,p.cd,NULL)) cd, AVG(p.swr) swr, AVG(p.reciprocity) cr, AVG(p.hie_norm) hie,
  COUNT(*) n
FROM `{P}.metrics_panel` p JOIN `{P}.sampling_frame_enriched` l USING(repo_id)
WHERE p.ym BETWEEN '{WIN_START}' AND '{WIN_END}' AND p.ym NOT IN UNNEST({GAP})
GROUP BY language, ym""")
lm_nrr = q(f"""
SELECT l.language, m.ym, AVG(IF(m.responses>0,1,0)) nrr, COUNT(*) nc
FROM `{P}.mdi_newcomers` m JOIN `{P}.sampling_frame_enriched` l USING(repo_id)
WHERE m.ym BETWEEN '{WIN_START}' AND '{MDI_END}' AND m.ym NOT IN UNNEST({GAP})
GROUP BY language, ym""")
lm = lm.merge(lm_nrr, on=["language","ym"], how="left")
# keep well-populated languages only (exclude 'unknown' and thin cells)
keep = [x for x in lm.language.value_counts().index if x!="unknown"][:14]
lm = lm[lm.language.isin(keep)].copy()
lm["high"]=lm.language.isin(HIGH_AI).astype(int)
lm["post"]=(lm.ym>=AI_YM).astype(int)
lm["t"]=ymidx(lm.ym)
print("languages in DiD:", sorted(lm.language.unique()))
print("high-AI:", HIGH_AI)


## 4. Difference-in-Differences — incremental AI-era effect
`y ~ C(language) + C(ym) + high:post`, SEs clustered by language. The **`high:post`** coefficient is the DiD: how much *more* the high-AI-exposure languages moved after 2022 than the comparison languages, net of common time shocks (COVID included via month FE) and fixed language differences.

In [ ]:
def did(metric):
    d=lm.dropna(subset=[metric]).copy()
    d["treatpost"]=d.high*d.post
    m=smf.ols(f"{metric} ~ C(language) + C(ym) + treatpost", data=d
             ).fit(cov_type="cluster", cov_kwds={"groups":d["language"]})
    return {"metric":metric,"did_coef":m.params["treatpost"],
            "p_value":m.pvalues["treatpost"], "n_lang":d.language.nunique()}
did_res=pd.DataFrame([did(x) for x in ["cd","swr","cr","hie","nrr"]]).round(5)
did_res.to_csv(f"{DRIVE_BASE}/tables/did_results.csv", index=False)
did_res


## 5. Placebo (fake 2021 break) + parallel-trends check
**Placebo:** restrict to pre-2022 data and pretend the treatment began at 2021-06 — the coefficient should be ~0 and non-significant. **Parallel-trends:** interact treatment with pre-period time — should be flat. If either fails for a metric, its DiD isn't trustworthy.

In [ ]:
def placebo(metric):
    d=lm[lm.ym < AI_YM].dropna(subset=[metric]).copy()
    d["ppost"]=(d.ym>=PLACEBO_YM).astype(int); d["ptreat"]=d.high*d.ppost
    m=smf.ols(f"{metric} ~ C(language) + C(ym) + ptreat", data=d
             ).fit(cov_type="cluster", cov_kwds={"groups":d["language"]})
    return {"metric":metric,"placebo_coef":m.params["ptreat"],"placebo_p":m.pvalues["ptreat"]}
def ptrend(metric):
    d=lm[lm.ym < AI_YM].dropna(subset=[metric]).copy()
    d["ht"]=d.high*d.t
    m=smf.ols(f"{metric} ~ C(language) + t + high + ht", data=d
             ).fit(cov_type="cluster", cov_kwds={"groups":d["language"]})
    return {"metric":metric,"pretrend_coef":m.params["ht"],"pretrend_p":m.pvalues["ht"]}
checks=pd.DataFrame([placebo(x) for x in ["cd","swr","cr","hie","nrr"]]).merge(
       pd.DataFrame([ptrend(x) for x in ["cd","swr","cr","hie","nrr"]]), on="metric").round(5)
checks.to_csv(f"{DRIVE_BASE}/tables/did_checks.csv", index=False)
checks


In [ ]:
print(its_res.to_string(index=False))
print(did_res.to_string(index=False))
print(checks.to_string(index=False))

## 6. Read-out
- **ITS** `ai_level`/`ai_slope`: an AI-era shift *beyond* secular + COVID trend. A significant one where the direction matches isolation (SWR↑, HIE↓, CR↓, NRR↓) is evidence of an AI-era acceleration.
- **DiD** `did_coef`: incremental effect on high-AI-exposure languages. Trust it **only if** the same metric's **placebo ≈ 0** and **pretrend ≈ 0**.
- Expect **NRR** to be the strongest AI-era candidate (it bent at 2022, not 2018). Expect **SWR/HIE** to show large *secular* slopes but weaker AI-specific effects. Wherever placebo/pretrend fail, report association only.

Results saved to `tables/its_results.csv`, `did_results.csv`, `did_checks.csv`.

Next: **06 networks** (enrichment) and **08 manual annotation** (validate that newcomer responses are genuinely mentoring). Then the manuscript.